In [ ]:
# === Colab-ready HumanEval evaluation (fixed for Phi-1.5 padding issue) ===

!pip install -q transformers datasets accelerate torch tqdm

import os, json, math, tempfile, subprocess, sys, time
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm
import pandas as pd

# ==== CONFIG ====
MODEL_NAME = "microsoft/phi-1_5"
NUM_SAMPLES = 5
MAX_NEW_TOKENS = 128
BATCH_SIZE = 8
TEMPERATURE = 0.8
TOP_P = 0.95
EVAL_TIMEOUT = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CONFIG:", MODEL_NAME, "NUM_SAMPLES=", NUM_SAMPLES, "BATCH_SIZE=", BATCH_SIZE, "DEVICE=", DEVICE)

# ==== SAFE EVAL ====
def eval_code_with_test(code_str: str, test_str: str, timeout: int = EVAL_TIMEOUT) -> bool:
    combined = code_str + "\n\n" + test_str + "\n"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as tf:
        tf.write(combined)
        fname = tf.name
    try:
        proc = subprocess.run([sys.executable, fname],
                              stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE,
                              timeout=timeout)
        return proc.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        try:
            os.remove(fname)
        except:
            pass

# ==== pass@k ====
def pass_at_k(n: int, c: int, k: int) -> float:
    if c == 0:
        return 0.0
    if k > n:
        k = n
    try:
        return 1.0 - math.comb(n - c, k) / math.comb(n, k)
    except ValueError:
        return 0.0

# ==== LOAD DATA ====
print("Loading HumanEval dataset …")
humaneval = load_dataset("openai_humaneval")["test"]
print("✅ Loaded", len(humaneval), "tasks")

# ==== LOAD MODEL ====
print(f"Loading model '{MODEL_NAME}' …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# FIX: add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None
)
# make sure model knows new padding
model.resize_token_embeddings(len(tokenizer))
model.eval()
print("✅ Model loaded on:", next(model.parameters()).device)

# ==== EVALUATION LOOP ====
def evaluate_model_on_humaneval(dataset, tokenizer, model,
                                batch_size=BATCH_SIZE, num_samples=NUM_SAMPLES,
                                max_new_tokens=MAX_NEW_TOKENS):
    results = []
    tasks = []
    for item in dataset:
        prompt = item.get("prompt") or item.get("code") or ""
        test = item.get("test") or item.get("ref") or ""
        task_id = item.get("task_id", None) or item.get("id", None) or len(tasks)
        tasks.append((task_id, prompt, test))

    for i in tqdm(range(0, len(tasks), batch_size), desc="Evaluating"):
        batch = tasks[i:i+batch_size]
        prompts = [p for (_id, p, t) in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(next(model.parameters()).device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_new_tokens=max_new_tokens,
                num_return_sequences=num_samples,
                pad_token_id=tokenizer.pad_token_id,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for idx_in_batch, (task_id, prompt, test_str) in enumerate(batch):
            start = idx_in_batch * num_samples
            end = start + num_samples
            completions = decoded[start:end]

            correct_count = 0
            for code in completions:
                if code.startswith(prompt):
                    code = code[len(prompt):].lstrip("\n")
                full_code = prompt + "\n" + code
                passed = eval_code_with_test(full_code, test_str, timeout=EVAL_TIMEOUT)
                if passed:
                    correct_count += 1

            results.append({
                "task_id": task_id,
                "correct_count": correct_count,
                "num_samples": num_samples,
                "pass@1": pass_at_k(num_samples, correct_count, 1),
                "pass@5": pass_at_k(num_samples, correct_count, 5),
                "pass@10": pass_at_k(num_samples, correct_count, 10)
            })

        time.sleep(0.05)
    return results

# ==== RUN EVAL ====
start_time = time.time()
results = evaluate_model_on_humaneval(humaneval, tokenizer, model)
elapsed = time.time() - start_time
print(f"✅ Evaluation complete in {elapsed/60:.2f} minutes, {len(results)} tasks done.")

# ==== SAVE + SUMMARY ====
df = pd.DataFrame(results)
summary = {
    "tasks": len(df),
    "avg_pass@1": df["pass@1"].mean(),
    "avg_pass@5": df["pass@5"].mean(),
    "avg_pass@10": df["pass@10"].mean()
}
print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))

out_csv = f"{MODEL_NAME.replace('/', '_')}_humaneval_results.csv"
df.to_csv(out_csv, index=False)
print("\nSaved results to:", out_csv)


CONFIG: microsoft/phi-1_5 NUM_SAMPLES= 5 BATCH_SIZE= 8 DEVICE= cuda
Loading HumanEval dataset …
✅ Loaded 164 tasks
Loading model 'microsoft/phi-1_5' …
✅ Model loaded on: cuda:0


Evaluating: 100%|██████████| 21/21 [05:15<00:00, 15.01s/it]

✅ Evaluation complete in 5.25 minutes, 164 tasks done.

=== SUMMARY ===
{
  "tasks": 164,
  "avg_pass@1": 0.20609756097560974,
  "avg_pass@5": 0.6829268292682927,
  "avg_pass@10": 0.6829268292682927
}

Saved results to: microsoft_phi-1_5_humaneval_results.csv
